<a href="https://colab.research.google.com/github/epi24/multimodal-meme-analysis/blob/main/multimodal_image_and_desc_NEW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import CLIPModel, CLIPProcessor
from torchvision import transforms
from tqdm.auto import tqdm
from PIL import Image
from collections import Counter
import gc

PATH_TRAIN_JSON = '/content/drive/MyDrive/meme_train.json'
PATH_VAL_JSON   = '/content/drive/MyDrive/meme_val.json'
PATH_IMAGES     = '/content/drive/MyDrive/all_memes/kym_memes'
PATH_DESCRIPTIONS        = '/content/drive/MyDrive/all_memes/Description_Text/kym_memes_description'
SAVE_DIR        = '/content/drive/MyDrive/multimodal_image_and_desc_NEW'
MODEL_NAME      = 'multimodal_model_image_and_desc'

CHECKPOINT_PATH = ''
TARGET_NUM_CLASSES = 250
# HYPERPARAMETER
EPOCHS        = 30
LEARNING_RATE = 1e-5
BATCH_SIZE = 2700
NUM_WORKERS = 12
PREFETCH_FACTOR = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

class MultimodalDataset(Dataset):
    def __init__(self, json_path, img_base_path, txt_base_path, processor, is_train=True):
        self.processor = processor
        self.img_base_path = img_base_path
        self.txt_base_path = txt_base_path
        self.samples = []
        self.description_cache = {}

        # Augmentation nur fürs Training
        if is_train:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
            ])
        else:
            self.transform = None

        print(f"--- [DATASET] Lade {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        # Map erstellen (Da die JSON gefiltert ist, sind das exakt deine 250 Klassen!)
        unique_labels = sorted(list(set(item['label'] for item in raw_data)))
        self.label_map = {label: idx for idx, label in enumerate(unique_labels)}
        self.id_to_label = {idx: label for label, idx in self.label_map.items()}

        # Map sofort abspeichern für die spatere Evaluation
        if is_train:
            with open(os.path.join(SAVE_DIR, f"{MODEL_NAME}_map.json"), 'w') as f:
                json.dump(self.id_to_label, f)

        skipped_count = 0

        for item in tqdm(raw_data, desc="Lade und verknüpfe Daten"):
            label_str = item.get('label')
            filename = item.get('filename')

            # Pfade zusammenbauen
            full_img_path = os.path.join(self.img_base_path, label_str, filename)
            txt_path = os.path.join(self.txt_base_path, f"{label_str}.txt")

            # Sicherheits-Check: Fehlt ausnahmsweise doch ein Bild oder ein Text?
            if not os.path.exists(full_img_path) or not os.path.exists(txt_path):
                skipped_count += 1
                continue

            # --- TEXT LADEN & ZENSIEREN (Anti-Data-Leakage) ---
            if label_str not in self.description_cache:
                with open(txt_path, 'r', encoding='utf-8', errors='ignore') as text_file:
                    raw_text = text_file.read().strip()

                    clean_text = raw_text.replace(label_str, "[MEME]")
                    clean_text = clean_text.replace(label_str.lower(), "[MEME]")
                    clean_text = clean_text.replace(label_str.upper(), "[MEME]")
                    clean_text = clean_text.replace(label_str.capitalize(), "[MEME]")

                    if len(clean_text) > 300: clean_text = clean_text[:300]
                    if len(clean_text) < 2: clean_text = "meme template"

                    self.description_cache[label_str] = clean_text

            self.samples.append({
                'img_path': full_img_path,
                'description': self.description_cache[label_str],
                'label': self.label_map[label_str]
            })

        print(f"-> Bereit: {len(self.samples)} gültige Paare geladen. (Übersprungen: {skipped_count})\n")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        try:
            image = Image.open(sample['img_path']).convert("RGB")
            if self.transform: image = self.transform(image)
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        img_inputs = self.processor(images=image, return_tensors="pt")
        text_inputs = self.processor(
            text=[sample['description']],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77
        )

        return {
            'pixel_values': img_inputs['pixel_values'].squeeze(0),
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

class MultimodalFusionNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

        # Backbone einfrieren
        for param in self.clip.parameters(): param.requires_grad = False

        # Letzte Schichten BEIDER Encoder auftauen (Partial Unfreezing)
        for param in self.clip.vision_model.encoder.layers[-1].parameters(): param.requires_grad = True
        for param in self.clip.text_model.encoder.layers[-1].parameters(): param.requires_grad = True
        for name, param in self.clip.named_parameters():
            if "layer_norm" in name: param.requires_grad = True

        # Classifier Head: 512 (Bild) + 512 (Text) = 1024
        self.fusion_head = nn.Sequential(
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        # 1. Bild-Features
        vision_out = self.clip.vision_model(pixel_values=pixel_values)
        img_embeds = self.clip.visual_projection(vision_out[1])

        # 2. Text-Features
        text_out = self.clip.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_embeds = self.clip.text_projection(text_out[1])

        # 3. Fusion (Zusammenkleben)
        combined = torch.cat((img_embeds, text_embeds), dim=1)

        # 4. Klassifizieren
        return self.fusion_head(combined)

def run_multimodal_training():
    print("--- Start Multimodal Training (Bild + Beschreibung) ---")

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    train_dataset = MultimodalDataset(PATH_TRAIN_JSON, PATH_IMAGES, PATH_DESCRIPTIONS, processor, is_train=True)
    val_dataset   = MultimodalDataset(PATH_VAL_JSON, PATH_IMAGES, PATH_DESCRIPTIONS, processor, is_train=False)

    num_classes = len(train_dataset.label_map)
    print(f"[INFO] Trainiere auf {num_classes} Klassen.")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=PREFETCH_FACTOR)
    val_loader   = DataLoader(val_dataset,batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=PREFETCH_FACTOR)

    model = MultimodalFusionNet(num_classes).to(DEVICE)

    # --- STANDARD OPTIMIZER (Alle Parameter lernen mit derselben Rate) ---
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')

    start_epoch = 0

    if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
        print(f"\n[INFO] Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            start_epoch = checkpoint["epoch"]
            print(f"[SUCCESS] Geladen! Starte ab Epoche {start_epoch+1}.")
        else:
            model.load_state_dict(checkpoint)
            print("[WARNUNG] Alter Checkpoint (Nur Gewichte).")

    print(f"\nStarte Training bis Epoche {EPOCHS}...\n")

    for epoch in range(start_epoch, EPOCHS):
        print(epoch)
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoche {epoch+1}/{EPOCHS} [Train]")

        for batch in pbar:
            optimizer.zero_grad()

            pixel_values = batch['pixel_values'].to(DEVICE)
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['label'].to(DEVICE)

            with torch.amp.autocast('cuda'):
                logits = model(pixel_values, input_ids, attention_mask)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoche {epoch+1} [Valid]", leave=False):
                pixel_values = batch['pixel_values'].to(DEVICE)
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['label'].to(DEVICE)

                with torch.amp.autocast('cuda'):
                    logits = model(pixel_values, input_ids, attention_mask)

                _, preds = torch.max(logits, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

                del pixel_values, input_ids, attention_mask, labels, logits

        val_acc = val_correct / val_total
        print(f" -> Resultat E{epoch+1}: Val Acc: {val_acc:.2%}")

        checkpoint_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }

        torch.save(checkpoint_dict, os.path.join(SAVE_DIR, f"{MODEL_NAME}_epoch_{epoch+1}.pth"))

if __name__ == "__main__":
    run_multimodal_training()



--- Start Multimodal Training (Bild + Beschreibung) ---


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

Epoche 29 [Valid]:   0%|          | 0/4 [00:00<?, ?it/s]

 -> Resultat E29: Val Acc: 100.00%
29


Epoche 30/30 [Train]:   0%|          | 0/29 [00:00<?, ?it/s]

Epoche 30 [Valid]:   0%|          | 0/4 [00:00<?, ?it/s]

 -> Resultat E30: Val Acc: 100.00%


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import CLIPModel, CLIPProcessor
from torchvision import transforms
from tqdm.auto import tqdm
from PIL import Image
import gc
import glob
import re

# --- KONFIGURATION ---
PATH_TRAIN_JSON = '/content/drive/MyDrive/meme_train.json'
PATH_VAL_JSON   = '/content/drive/MyDrive/meme_val.json'
PATH_IMAGES     = '/content/drive/MyDrive/all_memes/kym_memes'
PATH_DESCRIPTIONS = '/content/drive/MyDrive/all_memes/Description_Text/kym_memes_description'
SAVE_DIR        = '/content/drive/MyDrive/multimodal_image_and_desc_NEW'
MODEL_NAME      = 'multimodal_model_image_and_desc_2'

CHECKPOINT_PATH = ''
EPOCHS        = 20
LEARNING_RATE = 1e-5
# WICHTIG: Für Contrastive Learning sollte die Batch Size so groß wie möglich sein,
# aber 2700 ist oft zu groß für den VRAM. Passe das an deine GPU an!
BATCH_SIZE = 2700
NUM_WORKERS = 4
PREFETCH_FACTOR = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 1. DATASET
# ==========================================
class ContrastiveDataset(Dataset):
    def __init__(self, json_path, img_base_path, txt_base_path, processor, is_train=True):
        self.processor = processor
        self.img_base_path = img_base_path
        self.txt_base_path = txt_base_path
        self.samples = []
        self.description_cache = {}

        if is_train:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
            ])
        else:
            self.transform = None

        print(f"--- [DATASET] Lade {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        # Optional: Label-Map speichern (nützlich für spätere Eval)
        unique_labels = sorted(list(set(item['label'] for item in raw_data)))
        self.label_map = {label: idx for idx, label in enumerate(unique_labels)}
        self.id_to_label = {idx: label for label, idx in self.label_map.items()}

        if is_train:
            os.makedirs(SAVE_DIR, exist_ok=True)
            with open(os.path.join(SAVE_DIR, f"{MODEL_NAME}_map.json"), 'w') as f:
                json.dump(self.id_to_label, f)

        skipped_count = 0
        for item in tqdm(raw_data, desc="Lade und verknüpfe Daten"):
            label_str = item.get('label')
            filename = item.get('filename')

            full_img_path = os.path.join(self.img_base_path, label_str, filename)
            txt_path = os.path.join(self.txt_base_path, f"{label_str}.txt")

            if not os.path.exists(full_img_path) or not os.path.exists(txt_path):
                skipped_count += 1
                continue

            if label_str not in self.description_cache:
                with open(txt_path, 'r', encoding='utf-8', errors='ignore') as text_file:
                    raw_text = text_file.read().strip()
                    # Data Leak verhindern
                    clean_text = raw_text.replace(label_str, "[MEME]")
                    clean_text = clean_text.replace(label_str.lower(), "[MEME]")
                    clean_text = clean_text.replace(label_str.upper(), "[MEME]")
                    clean_text = clean_text.replace(label_str.capitalize(), "[MEME]")

                    if len(clean_text) > 300: clean_text = clean_text[:300]
                    if len(clean_text) < 2: clean_text = "meme template"
                    self.description_cache[label_str] = clean_text

            self.samples.append({
                'img_path': full_img_path,
                'description': self.description_cache[label_str]
            })

        print(f"-> Bereit: {len(self.samples)} gültige Paare geladen. (Übersprungen: {skipped_count})\n")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        try:
            image = Image.open(sample['img_path']).convert("RGB")
            if self.transform: image = self.transform(image)
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        img_inputs = self.processor(images=image, return_tensors="pt")
        text_inputs = self.processor(
            text=[sample['description']],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77
        )

        return {
            'pixel_values': img_inputs['pixel_values'].squeeze(0),
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0)
        }

# ==========================================
# 2. CONTRASTIVE LOSS (InfoNCE)
# ==========================================
class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, image_embeds, text_embeds):
        # Normalisieren
        image_embeds = F.normalize(image_embeds, p=2, dim=-1)
        text_embeds = F.normalize(text_embeds, p=2, dim=-1)

        # Ähnlichkeitsmatrix (Batch_Size x Batch_Size)
        logits = (image_embeds @ text_embeds.T) / self.temperature

        # Da jedes Bild im Batch zu seinem entsprechenden Text im Batch gehört,
        # ist die Diagonale unsere Zielklasse (Index 0 bis Batch_Size-1)
        batch_size = image_embeds.shape[0]
        targets = torch.arange(batch_size, device=image_embeds.device)

        # Symmetrische Cross-Entropy (sowohl Bild->Text als auch Text->Bild)
        loss_i2t = F.cross_entropy(logits, targets)
        loss_t2i = F.cross_entropy(logits.T, targets)

        return (loss_i2t + loss_t2i) / 2.0

# ==========================================
# 3. TRAINING LOOP
# ==========================================
def run_contrastive_training():
    global CHECKPOINT_PATH
    print("--- Start Contrastive Training (CLIP) ---")

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    train_dataset = ContrastiveDataset(PATH_TRAIN_JSON, PATH_IMAGES, PATH_DESCRIPTIONS, processor, is_train=True)
    val_dataset   = ContrastiveDataset(PATH_VAL_JSON, PATH_IMAGES, PATH_DESCRIPTIONS, processor, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # Direktes CLIP-Modell laden
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)

    # Optional: Backbone einfrieren und nur Head trainieren,
    # oder alles fine-tunen (bei Contrastive Learning oft besser, wenn man VRAM hat)
    # Für den Anfang tunen wir nur die letzten Schichten:
    for param in model.parameters(): param.requires_grad = False
    for param in model.vision_model.encoder.layers[-1].parameters(): param.requires_grad = True
    for param in model.text_model.encoder.layers[-1].parameters(): param.requires_grad = True
    for name, param in model.named_parameters():
        if "layer_norm" in name: param.requires_grad = True
    model.visual_projection.weight.requires_grad = True
    model.text_projection.weight.requires_grad = True

    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = ContrastiveLoss(temperature=0.07)
    scaler = torch.amp.GradScaler('cuda')

    # Automatische Checkpoint-Suche
    if not CHECKPOINT_PATH and os.path.exists(SAVE_DIR):
        pattern = os.path.join(SAVE_DIR, f"{MODEL_NAME}_epoch_*.pth")
        checkpoints = glob.glob(pattern)
        if checkpoints:
            try:
                CHECKPOINT_PATH = max(checkpoints, key=lambda x: int(re.search(r'epoch_(\d+)', x).group(1)))
                print(f"\n[INFO] Automatisch neuesten Checkpoint gefunden: {CHECKPOINT_PATH}")
            except:
                pass

    start_epoch = 0
    if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
        print(f"\n[INFO] Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            start_epoch = checkpoint["epoch"]
            print(f"[SUCCESS] Geladen! Starte ab Epoche {start_epoch+1}.")
        else:
            model.load_state_dict(checkpoint)

    print(f"\nStarte Training bis Epoche {EPOCHS}...\n")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoche {epoch+1}/{EPOCHS} [Train]")

        for batch in pbar:
            optimizer.zero_grad()

            pixel_values = batch['pixel_values'].to(DEVICE)
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)

            with torch.amp.autocast('cuda'):
                vision_out = model.vision_model(pixel_values=pixel_values)
                image_embeds = model.visual_projection(vision_out[1])

                text_out = model.text_model(input_ids=input_ids, attention_mask=attention_mask)
                text_embeds = model.text_projection(text_out[1])

                loss = criterion(image_embeds, text_embeds)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        # Validierung
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoche {epoch+1} [Valid]", leave=False):
                pixel_values = batch['pixel_values'].to(DEVICE)
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)

                with torch.amp.autocast('cuda'):
                    vision_out = model.vision_model(pixel_values=pixel_values)
                    image_embeds = model.visual_projection(vision_out[1])

                    text_out = model.text_model(input_ids=input_ids, attention_mask=attention_mask)
                    text_embeds = model.text_projection(text_out[1])

                    loss = criterion(image_embeds, text_embeds)
                    val_loss += loss.item()

                    # Accuracy Berechnung (In-Batch)
                    img_norm = F.normalize(image_embeds, p=2, dim=-1)
                    txt_norm = F.normalize(text_embeds, p=2, dim=-1)
                    logits = img_norm @ txt_norm.T

                    targets = torch.arange(img_norm.shape[0], device=DEVICE)
                    preds = torch.argmax(logits, dim=1)

                    val_correct += (preds == targets).sum().item()
                    val_total += targets.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_acc = val_correct / val_total if val_total > 0 else 0.0
        print(f" -> Resultat E{epoch+1}: Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2%}")

        checkpoint_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }
        os.makedirs(SAVE_DIR, exist_ok=True)
        torch.save(checkpoint_dict, os.path.join(SAVE_DIR, f"{MODEL_NAME}_epoch_{epoch+1}.pth"))

if __name__ == "__main__":
    run_contrastive_training() # Zum Ausführen einkommentieren
    pass


--- Start Contrastive Training (CLIP) ---


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

 -> Resultat E19: Train Loss: 3.7496 | Val Loss: 3.6944 | Val Acc: 7.45%


Epoche 20/20 [Train]:   0%|          | 0/29 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6a29b7a700>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6a29b7a700>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoche 20 [Valid]:   0%|          | 0/4 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6a29b7a700>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6a29b7a700>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

In [ ]:
import os
import json
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from tqdm.auto import tqdm
from PIL import Image
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd



CHECKPOINT_PATH   = '/content/drive/MyDrive/multimodal_image_and_desc_NEW/multimodal_model_image_and_desc_2_epoch_19.pth'
PATH_TEST_JSON    = '/content/drive/MyDrive/meme_test.json'
PATH_LABEL_MAP    = '/content/drive/MyDrive/multimodal_image_and_desc_NEW/multimodal_model_image_and_desc_map.json'
PATH_IMAGES       = '/content/drive/MyDrive/all_memes/kym_memes'
PATH_DESCRIPTIONS = '/content/drive/MyDrive/all_memes/Description_Text/kym_memes_description'

SAVE_DIR          = '/content/drive/MyDrive/multimodal_image_and_desc_NEW'
OUTPUT_CSV        = 'contrastive_evaluation_results2.csv'

BATCH_SIZE  = 2800  # Inferenz ist speichereffizienter, 64 ist ein sicherer Wert
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 2. DATASET (FÜR BILDER UND TEXT-ANKER)
# ==========================================
class EvalContrastiveDataset(Dataset):
    def __init__(self, json_path, img_base_path, txt_base_path, label_map_path, processor):
        self.processor = processor
        self.img_base_path = img_base_path
        self.txt_base_path = txt_base_path
        self.samples = []
        self.description_cache = {}

        print(f"Lade offizielle Label-Map: {label_map_path.split('/')[-1]}")
        with open(label_map_path, 'r', encoding='utf-8') as f:
            loaded_map = json.load(f)
            self.id_to_label = {int(k): v for k, v in loaded_map.items()}
            self.label_map = {v: int(k) for k, v in loaded_map.items()}

        print(f"--- [EVAL DATASET] Lade Test-Daten aus {json_path.split('/')[-1]}... ---")
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        for item in tqdm(raw_data, desc="Lade Daten"):
            label_str = item.get('label')

            if label_str not in self.label_map:
                continue

            filename = item.get('filename')
            full_img_path = os.path.join(self.img_base_path, label_str, filename)
            txt_path = os.path.join(self.txt_base_path, f"{label_str}.txt")

            if not os.path.exists(full_img_path) or not os.path.exists(txt_path):
                continue

            # --- TEXT LADEN & ZENSIEREN FÜR DIE ANKER ---
            if label_str not in self.description_cache:
                with open(txt_path, 'r', encoding='utf-8', errors='ignore') as text_file:
                    raw_text = text_file.read().strip()

                    clean_text = raw_text.replace(label_str, "[MEME]")
                    clean_text = clean_text.replace(label_str.lower(), "[MEME]")
                    clean_text = clean_text.replace(label_str.upper(), "[MEME]")
                    clean_text = clean_text.replace(label_str.capitalize(), "[MEME]")

                    if len(clean_text) > 300: clean_text = clean_text[:300]
                    if len(clean_text) < 2: clean_text = "meme template"

                    self.description_cache[label_str] = clean_text

            self.samples.append({
                'img_path': full_img_path,
                'label': self.label_map[label_str],
                'filename': filename
            })

        print(f"-> Bereit: {len(self.samples)} Test-Bilder geladen.\n")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        try:
            image = Image.open(sample['img_path']).convert("RGB")
        except:
            return self.__getitem__((idx + 1) % len(self.samples))

        img_inputs = self.processor(images=image, return_tensors="pt")

        return {
            'pixel_values': img_inputs['pixel_values'].squeeze(0),
            'label': torch.tensor(sample['label'], dtype=torch.long),
            'filename': sample['filename']
        }

# ==========================================
# 3. EVALUATION LOOP
# ==========================================
def run_contrastive_evaluation():
    print("--- Starte Contrastive Test-Evaluation ---")

    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    eval_dataset = EvalContrastiveDataset(PATH_TEST_JSON, PATH_IMAGES, PATH_DESCRIPTIONS, PATH_LABEL_MAP, processor)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    num_classes = len(eval_dataset.label_map)
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)

    if os.path.exists(CHECKPOINT_PATH):
        print(f"Lade Checkpoint: {CHECKPOINT_PATH}")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

        state_dict = checkpoint.get("model_state_dict", checkpoint)
        # Entferne das 'clip.' Präfix, das vom Training im MultimodalFusionNet stammt
        cleaned_state_dict = {k.replace('clip.', ''): v for k, v in state_dict.items()}

        model.load_state_dict(cleaned_state_dict, strict=False)
        print("[SUCCESS] Gewichte erfolgreich geladen.")
    else:
        print(f"[ERROR] Checkpoint nicht gefunden: {CHECKPOINT_PATH}")
        return

    model.eval()

    results_list = []
    all_preds = []
    all_labels = []

    print("Erzeuge statische Text-Anker aus den Beschreibungen...")
    # Wir sortieren die Beschreibungen exakt nach ihrer ID (0 bis 249)
    ordered_descriptions = [eval_dataset.description_cache[eval_dataset.id_to_label[i]] for i in range(num_classes)]

    with torch.no_grad():
        # Text-Embeddings einmalig berechnen und normalisieren
        text_inputs = processor(text=ordered_descriptions, return_tensors="pt", padding=True, truncation=True, max_length=77).to(DEVICE)
        text_outputs = model.text_model(**text_inputs)
        static_text_embeds = model.text_projection(text_outputs[1])
        static_text_embeds = F.normalize(static_text_embeds, p=2, dim=-1)

        print("Berechne Vorhersagen auf ungesehenen Test-Bildern...")

        for batch in tqdm(eval_loader):
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            filenames = batch['filename']

            with torch.amp.autocast('cuda'):
                # Bild-Embeddings berechnen und normalisieren
                vision_outputs = model.vision_model(pixel_values=pixel_values)
                image_embeds = model.visual_projection(vision_outputs[1])
                image_embeds = F.normalize(image_embeds, p=2, dim=-1)

                # Ähnlichkeit berechnen (Dot Product)
                logits = 100.0 * (image_embeds @ static_text_embeds.T)

            probs = torch.softmax(logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i in range(len(filenames)):
                pred_idx = preds[i].item()
                true_idx = labels[i].item()

                results_list.append({
                    "Dateiname": filenames[i],
                    "Wahre Klasse": eval_dataset.id_to_label[true_idx],
                    "Vorhersage": eval_dataset.id_to_label[pred_idx],
                    "Status": "KORREKT" if pred_idx == true_idx else "FALSCH",
                    "Sicherheit_Prozent": round(confidences[i].item() * 100, 2)
                })

    # Gesamte Accuracy
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n========================================")
    print(f"CONTRASTIVE MODEL ACCURACY (Test Set): {acc:.2%}")
    print(f"========================================\n")

    # Metriken berechnen
    class_names = [eval_dataset.id_to_label[i] for i in range(num_classes)]
    report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)

    # Detail-CSV speichern
    df_details = pd.DataFrame(results_list)
    save_path_csv = os.path.join(SAVE_DIR, OUTPUT_CSV)
    df_details.to_csv(save_path_csv, index=False, sep=';', encoding='utf-8-sig')

    # Metriken-CSV speichern
    metrics_list = []
    for name in class_names:
        metrics = report_dict[name]
        metrics_list.append({
            "Meme": name,
            "Precision": round(metrics['precision'], 2),
            "Recall": round(metrics['recall'], 2),
            "F1-Score": round(metrics['f1-score'], 2),
            "Anzahl": metrics['support']
        })

    df_metrics = pd.DataFrame(metrics_list)
    df_metrics = df_metrics.sort_values(by="F1-Score", ascending=True)
    df_metrics.to_csv(os.path.join(SAVE_DIR, "contrastive_metrics.csv"), index=False, sep=';')

    print("[FERTIG] Tabellen gespeichert.")

if __name__ == "__main__":
    run_contrastive_evaluation()


--- Starte Contrastive Test-Evaluation ---


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Lade Checkpoint: /content/drive/MyDrive/multimodal_image_and_desc_NEW/multimodal_model_image_and_desc_2_epoch_19.pth
[SUCCESS] Gewichte erfolgreich geladen.
Erzeuge statische Text-Anker aus den Beschreibungen...
Berechne Vorhersagen auf ungesehenen Test-Bildern...


  0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in by


CONTRASTIVE MODEL ACCURACY (Test Set): 72.96%

[FERTIG] Tabellen gespeichert.
